# Contourdata voorbereiden

Dit notebook zet ruwe contourdata om naar invoerbestanden voor het dashboard (`input/lden_contour.csv` en `input/lnight_contour.csv`).

Per geluidscontour (1 dB-band) berekenen we onder meer bevolking, woningen, dosis–effect, vastgoedstock en maatregelstromen.

Zie **§0** voor een volledige inventaris van alle bronbestanden en **§5.1** voor het geconsolideerde contour-overzicht.

## Overzicht bronbestanden en kolommen

### Vlaanderen (+ rest land) — `data/contour_vlaanderen_stocks.xlsx`

Eén rij per **1 dB-contourband** (bijv. `45-46`). Na hernoemen (`hernoem_vlaanderen_kolommen`):

| Kolom | Beschrijving |
|-------|----------------|
| `geluidscontour` | Label van de band (tekst, bijv. `45-46`) |
| `db_ondergrens` | Ondergrens van de band in dB |
| `db_bovengrens` | Bovengrens van de band in dB |
| `inwoners` | Inwoners in de contour (buiten Brussel) |
| `aantal_woningen` | Woningen in de contour (buiten Brussel) |
| `aantal_bebouwbare_percelen_woongebied` | Bebouwbare percelen door woongebied-aanwijzing (laatste 5 jaar) |
| `aantal_niet_bebouwbare_percelen_woongebied_schrapping` | Niet-bebouwbare percelen door schrapping woongebied |
| `prijs_onbebouwde_bebouwbare_percelen` | Prijs onbebouwd bebouwbare percelen |
| `prijs_onbebouwde_onbebouwbare_percelen` | Prijs onbebouwd onbebouwbare percelen |
| `prijs_bewoonde_niet_geïsoleerde_woning` | Prijs bewoonde niet-geïsoleerde woning |
| `prijs_bewoonde_geïsoleerde_woning` | Prijs bewoonde geïsoleerde woning |

### Brussel — `data/inwoners_brussel_sector_contour_2024.xlsx`

Eerst per statistische sector, daarna **gesommeerd per dB**. Na hernoemen (`hernoem_brussel_kolommen`):

| Kolom | Beschrijving |
|-------|----------------|
| `db` | Geluidsniveau van de contour (enkelvoudig dB-getal, geen band) |
| `inwoners` | Inwoners in de contour (`Population dans le contour`) |

### Landelijk totaal — `df_lden` / `df_lnight` (na samenvoegen)

| Kolom | Beschrijving |
|-------|----------------|
| *(alle Vlaanderen-kolommen)* | Behouden; prijzen en percelen komen alleen uit Vlaanderen |
| `inwoners_vlaanderen` | Inwoners buiten Brussel (oorspronkelijke waarde) |
| `inwoners_brussel` | Brusselse inwoners gekoppeld aan de band (`db` = `db_ondergrens`) |
| `inwoners` | `inwoners_vlaanderen` + `inwoners_brussel` |
| `gemiddeld_aantal_inwoners_per_huis` | Ratio buiten Brussel (voor schatting woningen Brussel) |
| `aantal_woningen_vlaanderen` | Woningen buiten Brussel |
| `aantal_woningen_brussel` | Geschatte woningen Brussel (`inwoners_brussel` / gemiddelde bezetting) |
| `aantal_woningen_totaal` | Som Vlaanderen + Brussel |
| `aantal_ernstig_gehinderden_vlaanderen_2026` | Ernstig gehinderden buiten Brussel |
| `aantal_ernstig_gehinderden_brussel_2026` | Ernstig gehinderden in Brussel |
| `aantal_ernstig_gehinderden_totaal_2026` | Som Vlaanderen + Brussel |
| `aantal_<stock>_vlaanderen_2026` / `_brussel_2026` / `_totaal_2026` | Vastgoedstock per regio; simulatie op `*_totaal_*` |


In [22]:
import pandas as pd

from config import BEGINJAAR


def maak_regio_kolommen(
    df: pd.DataFrame,
    basis: str,
    jaar: int,
    vlaanderen: pd.Series,
    brussel: pd.Series,
) -> None:
    """Schrijf {basis}_vlaanderen_{jaar}, {basis}_brussel_{jaar}, {basis}_totaal_{jaar}."""
    df[f"{basis}_vlaanderen_{jaar}"] = vlaanderen
    df[f"{basis}_brussel_{jaar}"] = brussel
    df[f"{basis}_totaal_{jaar}"] = vlaanderen + brussel


def hernoem_vlaanderen_kolommen(df: pd.DataFrame) -> pd.DataFrame:
    """Vertaal en verkort kolomnamen uit contour_vlaanderen_stocks.xlsx naar duidelijk Nederlands."""
    mapping = {
        "db_contour": "geluidscontour",
        "lower": "db_ondergrens",
        "upper": "db_bovengrens",
        "huizen": "aantal_woningen",
    }
    for kolom in df.columns:
        if "bebouwbare percelen die werden" in kolom:
            mapping[kolom] = "aantal_bebouwbare_percelen_woongebied"
        elif "niet-bebouwbare" in kolom or "woongebied te schrappen" in kolom:
            mapping[kolom] = "aantal_niet_bebouwbare_percelen_woongebied_schrapping"
    return df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})


def hernoem_brussel_kolommen(df: pd.DataFrame) -> pd.DataFrame:
    """Vertaal Franse kolomnamen uit de Brussel-populatiedata."""
    return df.rename(
        columns={
            "dB": "db",
            "Population dans le contour": "inwoners",
        }
    )


def samenvoegen_vlaanderen_brussel(
    df_vlaanderen: pd.DataFrame,
    df_brussel: pd.DataFrame,
) -> pd.DataFrame:
    """
    Voeg landelijke contourdata samen met Brusselse inwoners.

    Brussel heeft één dB-waarde per rij; Vlaanderen heeft banden [db_ondergrens, db_bovengrens].
    Koppeling: Brusselse inwoners op niveau db worden toegevoegd aan de band waar db_ondergrens == db
    (bijv. Brussel 45 → band 45–46).
    """
    df = df_vlaanderen.copy()
    inwoners_per_db = df_brussel.set_index("db")["inwoners"]

    df["inwoners_vlaanderen"] = df["inwoners"]
    df["inwoners_brussel"] = df["db_ondergrens"].map(inwoners_per_db).fillna(0)
    df["inwoners"] = df["inwoners_vlaanderen"] + df["inwoners_brussel"]

    gemiddeld = df["gemiddeld_aantal_inwoners_per_huis"]
    df["aantal_woningen_vlaanderen"] = df["aantal_woningen"]
    df["aantal_woningen_brussel"] = df["inwoners_brussel"] / gemiddeld
    df["aantal_woningen_totaal"] = (
        df["aantal_woningen_vlaanderen"] + df["aantal_woningen_brussel"]
    )
    df = df.drop(columns=["aantal_woningen"])

    return df


## 0. Inventaris alle databronnen

Overzicht van alle bestanden in `data/` die relevant zijn voor contourberekeningen en flow rates. Naamgevingsconventie: zie `data/README.md`.

| Categorie | Bestand(en) | Ruimtelijk niveau | Gebruik in dit notebook |
|-----------|-------------|-------------------|-------------------------|
| **Contour Lden/Lnight Vlaanderen** | `contour_vlaanderen_stocks.xlsx` (`lden`, `lnight`) | 1 dB-band (bijv. 45–46) | §1 — basis `df_lden` / `df_lnight` |
| **Inwoners Brussel** | `inwoners_brussel_sector_contour_2024.xlsx` | Statistische sector × dB | §3 — optellen inwoners; §10 — conversietabel gemeente→dB |
| **Vergunningen** | `vergunningen_*_2026.xlsx` / `*_lang.csv` | Gemeente × jaar | §9–§11 — flow-tellers |
| **Transacties** | `transacties_vastgoed/transacties_*.csv` | CaPaKey (perceel) | Flow rates aankoop/voorkoop (later: CaPaKey→contour) |

Onderstaande cel scant `data/` en toont formaat, omvang en kolommen per bron.

In [23]:
from pathlib import Path

DATA = Path("data")

BRONNEN = [
    {"categorie": "Contour Vlaanderen", "pad": DATA / "contour_vlaanderen_stocks.xlsx", "type": "excel_lden_lnight"},
    {"categorie": "Inwoners Brussel", "pad": DATA / "inwoners_brussel_sector_contour_2024.xlsx", "type": "excel_lden_lnight"},
    {"categorie": "Vergunningen omgevingsloket", "pad": DATA / "vergunningen_omgevingsloket_2026_lang.csv", "type": "csv_lang"},
    {"categorie": "Vergunningen kwetsbare functies", "pad": DATA / "vergunningen_kwetsbare_functies_2026_lang.csv", "type": "csv_lang"},
    {"categorie": "Vergunningen verkaveling", "pad": DATA / "vergunningen_verkaveling_2026_lang.csv", "type": "csv_lang"},
]

for p in sorted((DATA / "transacties_vastgoed").glob("transacties_*.csv")):
    BRONNEN.append({"categorie": "Transacties", "pad": p, "type": "csv_tab"})


def beschrijf_bron(entry: dict) -> dict:
    pad = entry["pad"]
    info = {"categorie": entry["categorie"], "bestand": str(pad), "bestaat": pad.exists()}
    if not pad.exists():
        return info
    if entry["type"] == "excel_lden_lnight":
        xl = pd.ExcelFile(pad)
        info["tabbladen"] = xl.sheet_names
        for sheet in xl.sheet_names:
            df = pd.read_excel(pad, sheet_name=sheet, nrows=3)
            info[f"kolommen_{sheet}"] = list(df.columns)
            info[f"rijen_{sheet}"] = len(pd.read_excel(pad, sheet_name=sheet))
    elif entry["type"] == "csv_lang":
        df = pd.read_csv(pad, sep=";", nrows=5)
        info["kolommen"] = list(df.columns)
        info["rijen"] = sum(1 for _ in open(pad, encoding="utf-8")) - 1
    elif entry["type"] == "csv_tab":
        df = pd.read_csv(pad, sep="\t", nrows=5)
        info["kolommen"] = list(df.columns)
        info["rijen"] = sum(1 for _ in open(pad, encoding="utf-8")) - 1
    return info


df_inventaris = pd.DataFrame([beschrijf_bron(b) for b in BRONNEN])
display(df_inventaris)

,categorie,bestand,bestaat,tabbladen,kolommen_lden,rijen_lden,kolommen_lnight,rijen_lnight,kolommen,rijen
0,Contour Vlaanderen,data\contour_vlaanderen_stocks.xlsx,True,"[lden, lnight]","[db_contour, lower, upper, inwoners, huizen, a...",30.0,"[db_contour, lower, upper, inwoners, huizen, a...",30.0,NaN,NaN
1,Inwoners Brussel,data\inwoners_brussel_sector_contour_2024.xlsx,True,"[lden, lnight]","[fid, CS01012022, T_SEC_NL, T_SEC_FR, T_SEC_DE...",1033.0,"[fid, CS01012022, T_SEC_NL, T_SEC_FR, T_SEC_DE...",613.0,NaN,NaN
2,Vergunningen omgevingsloket,data\vergunningen_omgevingsloket_2026_lang.csv,True,NaN,NaN,NaN,NaN,NaN,"[bron, jaar_indiening, gemeente, handeling, ge...",24852.0
3,Vergunningen kwetsbare functies,data\vergunningen_kwetsbare_functies_2026_lang...,True,NaN,NaN,NaN,NaN,NaN,"[bron, jaar_indiening, gemeente, handeling, ge...",1332.0
4,Vergunningen verkaveling,data\vergunningen_verkaveling_2026_lang.csv,True,NaN,NaN,NaN,NaN,NaN,"[bron, jaar_indiening, gemeente, handeling, ge...",4530.0
5,Transacties,data\transacties_vastgoed\transacties_appartem...,True,NaN,NaN,NaN,NaN,NaN,"[NISCode, sum_ParcelsNumber, avg_PriceP25, avg...",19790.0
6,Transacties,data\transacties_vastgoed\transacties_handel.csv,True,NaN,NaN,NaN,NaN,NaN,"[NISCode, sum_ParcelsNumber, avg_PriceP25, avg...",19790.0
7,Transacties,data\transacties_vastgoed\transacties_industri...,True,NaN,NaN,NaN,NaN,NaN,"[NISCode, sum_ParcelsNumber, avg_PriceP25, avg...",19790.0
8,Transacties,data\transacties_vastgoed\transacties_industri...,True,NaN,NaN,NaN,NaN,NaN,"[NISCode, sum_ParcelsNumber, avg_PriceP25, avg...",19790.0
9,Transacties,data\transacties_vastgoed\transacties_kantoren...,True,NaN,NaN,NaN,NaN,NaN,"[NISCode, sum_ParcelsNumber, avg_PriceP25, avg...",19790.0


## 1. Landelijke contourgegevens laden

Bron: `data/contour_vlaanderen_stocks.xlsx` (tabbladen `lden` en `lnight`).

Elke rij is één **1 dB-geluidscontour** met onder meer grenzen van de band, inwoners, woningen en prijzen. Kolomnamen worden meteen hernoemd naar duidelijk Nederlands zodat de rest van het notebook leesbaar blijft.


In [24]:
df_lden_vlaanderen = pd.read_excel("data/contour_vlaanderen_stocks.xlsx", sheet_name="lden")
df_lnight_vlaanderen = pd.read_excel("data/contour_vlaanderen_stocks.xlsx", sheet_name="lnight")

df_lden_vlaanderen = hernoem_vlaanderen_kolommen(df_lden_vlaanderen)
df_lnight_vlaanderen = hernoem_vlaanderen_kolommen(df_lnight_vlaanderen)

df_lden_vlaanderen.head()

,geluidscontour,db_ondergrens,db_bovengrens,inwoners,aantal_woningen,aantal_bebouwbare_percelen_woongebied,aantal_niet_bebouwbare_percelen_woongebied_schrapping,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning
0,45-46,45,46,45987,17688,0.6,4.4,200000,100000,400000,500000
1,46-47,46,47,28960,11714,1,4,197500,97500,395000,495000
2,47-48,47,48,27769,10989,0.6,2,195000,95000,390000,490000
3,48-49,48,49,38500,12806,0.2,3,192500,92500,385000,485000
4,49-50,49,50,26406,9891,0,2.8,190000,90000,380000,480000


## 2. Ontbrekende waarden opschonen

In de bron staan soms tekstwaarden `geen data`. Die vervangen we door `0` zodat berekeningen niet falen.


In [25]:
df_lden_vlaanderen.replace("geen data", 0, inplace=True)
df_lnight_vlaanderen.replace("geen data", 0, inplace=True)


,geluidscontour,db_ondergrens,db_bovengrens,inwoners,aantal_woningen,aantal_bebouwbare_percelen_woongebied,aantal_niet_bebouwbare_percelen_woongebied_schrapping,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning
0,40-41,40,41,35414,12046,0.2,2.4,200000,100000,400000,500000
1,41-42,41,42,31633,11679,0,2.4,197500,97500,395000,495000
2,42-43,42,43,32185,11252,0,3.4,195000,95000,390000,490000
3,43-44,43,44,25275,8657,0,2.8,192500,92500,385000,485000
4,44-45,44,45,29155,9613,0,2.8,190000,90000,380000,480000
5,45-46,45,46,33567,11061,0,3.8,187500,87500,375000,475000
6,46-47,46,47,23487,7957,0,1.6,185000,85000,370000,470000
7,47-48,47,48,15552,5362,0.2,2.8,182500,82500,365000,465000
8,48-49,48,49,10335,3538,0.2,1.2,180000,80000,360000,460000
9,49-50,49,50,8694,2969,0.2,0.6,177500,77500,355000,455000


## 3. Bevolking Brussels Hoofdstedelijk Gewest

Bron: `data/inwoners_brussel_sector_contour_2024.xlsx`.

De data staan per **statistische sector**; dezelfde dB-waarde komt dus meerdere keren voor. We sommeren `inwoners` per `db` zodat één rij overblijft per geluidsniveau — analoog aan de landelijke contourbanden.


In [26]:
df_lden_brussels = pd.read_excel(
    "data/inwoners_brussel_sector_contour_2024.xlsx",
    sheet_name="lden",
)
df_lnight_brussels = pd.read_excel(
    "data/inwoners_brussel_sector_contour_2024.xlsx",
    sheet_name="lnight",
)

df_lden_brussels = hernoem_brussel_kolommen(
    df_lden_brussels[["dB", "Population dans le contour"]]
)
df_lnight_brussels = hernoem_brussel_kolommen(
    df_lnight_brussels[["dB", "Population dans le contour"]]
)

In [27]:
df_lden_brussels = df_lden_brussels.groupby("db", as_index=False)["inwoners"].sum()
df_lnight_brussels = df_lnight_brussels.groupby("db", as_index=False)["inwoners"].sum()

df_lden_brussels.head()

,db,inwoners
0,45,105153.667771
1,46,127964.973017
2,47,94698.794036
3,48,111977.055565
4,49,103292.390429


## 4. Schatting aantal woningen in het Brussels Hoofdstedelijk Gewest

Voor Brussel ontbreken woningaantallen per sector. We schatten het aantal woningen per geluidscontour via het **gemiddeld aantal inwoners per woning** buiten Brussel (`inwoners` / `aantal_woningen` in `contour_vlaanderen_stocks.xlsx`).

**Stap 1 — gemiddelde bezetting buiten Brussel** (per contourband):

$$
\bar{i}_{\text{buiten Brussel}} = \frac{I_{\text{buiten Brussel}}}{W_{\text{buiten Brussel}}}
$$

**Stap 2 — geschat aantal woningen per contour** (Brussel + rest van het land):

$$
\hat{W}_{\text{contour}} = \frac{I_{\text{Brussel}} + I_{\text{buiten Brussel}}}{\bar{i}_{\text{buiten Brussel}}}
$$

| Symbool | Kolom in dit notebook |
|--------|------------------------|
| $I_{\text{Brussel}}$ | `inwoners` in `df_*_brussels` |
| $I_{\text{buiten Brussel}}$ | `inwoners` in `df_lden_vlaanderen` / `df_lnight_vlaanderen` |
| $W_{\text{buiten Brussel}}$ | `aantal_woningen` in de Vlaanderen-dataframes |
| $\hat{W}_{\text{contour}}$ | `aantal_woningen_totaal` na `samenvoegen_vlaanderen_brussel` (§5) |


In [28]:
df_lden_vlaanderen["gemiddeld_aantal_inwoners_per_huis"] = (
    df_lden_vlaanderen["inwoners"] / df_lden_vlaanderen["aantal_woningen"]
)
df_lnight_vlaanderen["gemiddeld_aantal_inwoners_per_huis"] = (
    df_lnight_vlaanderen["inwoners"] / df_lnight_vlaanderen["aantal_woningen"]
)

## 5. Samenvoegen Vlaanderen en Brussel

Vlaanderen levert de volledige contourtabel (woningen, prijzen, percelen). Brussel levert enkel **inwoners per dB**.

- **Inwoners:** per band optellen (`inwoners_vlaanderen` + `inwoners_brussel`).
- **Woningen:** buiten Brussel blijven de gemeten waarden; voor Brussel schatten we woningen met de gemiddelde bezetting buiten Brussel (zie §4).
- **Koppeling dB:** `db` (Brussel) = `db_ondergrens` (Vlaanderen).

Het resultaat is `df_lden` / `df_lnight` voor alle verdere stappen.

In [29]:
df_lden = samenvoegen_vlaanderen_brussel(df_lden_vlaanderen, df_lden_brussels)
df_lnight = samenvoegen_vlaanderen_brussel(df_lnight_vlaanderen, df_lnight_brussels)

df_lden[
    [
        "geluidscontour",
        "db_ondergrens",
        "db_bovengrens",
        "inwoners_vlaanderen",
        "inwoners_brussel",
        "inwoners",
        "aantal_woningen_vlaanderen",
        "aantal_woningen_brussel",
        "aantal_woningen_totaal",
    ]
].head()

,geluidscontour,db_ondergrens,db_bovengrens,inwoners_vlaanderen,inwoners_brussel,inwoners,aantal_woningen_vlaanderen,aantal_woningen_brussel,aantal_woningen_totaal
0,45-46,45,46,45987,105153.667771,151140.667771,17688,40445.301401,58133.301401
1,46-47,46,47,28960,127964.973017,156924.973017,11714,51760.417608,63474.417608
2,47-48,47,48,27769,94698.794036,122467.794036,10989,37475.063836,48464.063836
3,48-49,48,49,38500,111977.055565,150477.055565,12806,37246.186326,50052.186326
4,49-50,49,50,26406,103292.390429,129698.390429,9891,38690.639769,48581.639769


### 5.1 Geconsolideerd overzicht Lden en Lnight

Na samenvoegen van Vlaanderen en Brussel bevat elke rij één **1 dB-contourband** met alle beschikbare variabelen. Onderstaande tabellen tonen de kolomstructuur en kerncijfers per band.

In [30]:
def overzicht_contour(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Kerncijfers per geluidscontour voor documentatie."""
    kern = [
        "geluidscontour", "db_ondergrens", "db_bovengrens",
        "inwoners_vlaanderen", "inwoners_brussel", "inwoners",
        "aantal_woningen_vlaanderen", "aantal_woningen_brussel", "aantal_woningen_totaal",
        "aantal_bebouwbare_percelen_woongebied",
    ]
    kolommen = [c for c in kern if c in df.columns]
    print(f"=== {label} — {len(df)} contourbanden, {len(df.columns)} kolommen ===")
    print("Kolommen:", ", ".join(df.columns))
    return df[kolommen].copy()


display(overzicht_contour(df_lden, "Lden"))
display(overzicht_contour(df_lnight, "Lnight"))

=== Lden — 30 contourbanden, 16 kolommen ===
Kolommen: geluidscontour, db_ondergrens, db_bovengrens, inwoners, aantal_bebouwbare_percelen_woongebied, aantal_niet_bebouwbare_percelen_woongebied_schrapping, prijs_onbebouwde_bebouwbare_percelen, prijs_onbebouwde_onbebouwbare_percelen, prijs_bewoonde_niet_geïsoleerde_woning, prijs_bewoonde_geïsoleerde_woning, gemiddeld_aantal_inwoners_per_huis, inwoners_vlaanderen, inwoners_brussel, aantal_woningen_vlaanderen, aantal_woningen_brussel, aantal_woningen_totaal


,geluidscontour,db_ondergrens,db_bovengrens,inwoners_vlaanderen,inwoners_brussel,inwoners,aantal_woningen_vlaanderen,aantal_woningen_brussel,aantal_woningen_totaal,aantal_bebouwbare_percelen_woongebied
0,45-46,45,46,45987,105153.667771,151140.667771,17688,40445.301401,58133.301401,0.6
1,46-47,46,47,28960,127964.973017,156924.973017,11714,51760.417608,63474.417608,1
2,47-48,47,48,27769,94698.794036,122467.794036,10989,37475.063836,48464.063836,0.6
3,48-49,48,49,38500,111977.055565,150477.055565,12806,37246.186326,50052.186326,0.2
4,49-50,49,50,26406,103292.390429,129698.390429,9891,38690.639769,48581.639769,0
5,50-51,50,51,35323,73998.639084,109321.639084,12064,25273.039717,37337.039717,0
6,51-52,51,52,25892,46161.300762,72053.300762,8936,15931.460822,24867.460822,0
7,52-53,52,53,24298,38721.081206,63019.081206,8374,13344.733477,21718.733477,0
8,53-54,53,54,35767,45254.076985,81021.076985,11579,14650.290978,26229.290978,0
9,54-55,54,55,26418,28920.855102,55338.855102,8838,9675.316731,18513.316731,0


=== Lnight — 30 contourbanden, 16 kolommen ===
Kolommen: geluidscontour, db_ondergrens, db_bovengrens, inwoners, aantal_bebouwbare_percelen_woongebied, aantal_niet_bebouwbare_percelen_woongebied_schrapping, prijs_onbebouwde_bebouwbare_percelen, prijs_onbebouwde_onbebouwbare_percelen, prijs_bewoonde_niet_geïsoleerde_woning, prijs_bewoonde_geïsoleerde_woning, gemiddeld_aantal_inwoners_per_huis, inwoners_vlaanderen, inwoners_brussel, aantal_woningen_vlaanderen, aantal_woningen_brussel, aantal_woningen_totaal


,geluidscontour,db_ondergrens,db_bovengrens,inwoners_vlaanderen,inwoners_brussel,inwoners,aantal_woningen_vlaanderen,aantal_woningen_brussel,aantal_woningen_totaal,aantal_bebouwbare_percelen_woongebied
0,40-41,40,41,35414,99777.787263,135191.787263,12046,33939.211198,45985.211198,0.2
1,41-42,41,42,31633,95059.230445,126692.230445,11679,35096.157568,46775.157568,0
2,42-43,42,43,32185,77511.734810,109696.734810,11252,27098.401121,38350.401121,0
3,43-44,43,44,25275,51448.950425,76723.950425,8657,17621.901635,26278.901635,0
4,44-45,44,45,29155,46936.946915,76091.946915,9613,15476.071710,25089.071710,0
5,45-46,45,46,33567,23871.855346,57438.855346,11061,7866.255310,18927.255310,0
6,46-47,46,47,23487,13219.675222,36706.675222,7957,4478.603302,12435.603302,0
7,47-48,47,48,15552,10340.191508,25892.191508,5362,3565.078888,8927.078888,0.2
8,48-49,48,49,10335,3881.450569,14216.450569,3538,1328.744278,4866.744278,0.2
9,49-50,49,50,8694,1741.137966,10435.137966,2969,594.598415,3563.598415,0.2


## 5. Dosis–effect en ernstig gehinderden

Voor elke contour nemen we het **midden van de dB-band** (`db_midden`) en passen daarop de dosis–effectformule toe (andere coëfficiënten voor Lden en Lnight).

Het aantal ernstig gehinderden per regio is `inwoners × dosis_effect_relatie` (zelfde relatie per band). Het totaal is de som van Vlaanderen+land en Brussel.


### Benodigde outputkolommen (checklist)

- [x] Geluidscontouren en dB (`geluidscontour`, `db_ondergrens`, `db_bovengrens`)
- [x] Aantal inwoners per geluidscontour
- [x] Aantal woningen per geluidscontour (`aantal_woningen_vlaanderen`, `_brussel`, `_totaal`)
- [x] Gemiddeld aantal inwoners per woning
- [x] Dosis-effectrelatie (Lden vs Lnight)
- [x] Aantal ernstig gehinderden
- [x] Bebouwbare percelen door woongebied (`aantal_bebouwbare_percelen_woongebied`)
- [ ] Prijzen en overige vastgoedstocks (deels voorbeelddata hieronder)


In [31]:
def dosis_effect_relatie(db, teller_1, teller_2, teller_3, noemer):
    return (teller_1 + teller_2 * db + pow(db, 2) * teller_3) / noemer


def lden_dosis_effect_relatie(db):
    return dosis_effect_relatie(db, -50.9693, 1.0168, 0.0072, 100)


def lnight_dosis_effect_relatie(db):
    return dosis_effect_relatie(db, 16.7885, -0.9293, 0.0198, 100)

In [32]:
df_lden["db_midden"] = (df_lden["db_ondergrens"] + df_lden["db_bovengrens"]) / 2
df_lnight["db_midden"] = (df_lnight["db_ondergrens"] + df_lnight["db_bovengrens"]) / 2

df_lden["dosis_effect_relatie"] = df_lden["db_midden"].apply(lden_dosis_effect_relatie)
df_lnight["dosis_effect_relatie"] = df_lnight["db_midden"].apply(lnight_dosis_effect_relatie)

In [33]:
for df in (df_lden, df_lnight):
    maak_regio_kolommen(
        df,
        "aantal_ernstig_gehinderden",
        BEGINJAAR,
        df["inwoners_vlaanderen"] * df["dosis_effect_relatie"],
        df["inwoners_brussel"] * df["dosis_effect_relatie"],
    )

## 6. Vastgoedstock (voorbeelddata)

De onderstaande aantallen zijn **plaatsvervangers** op basis van `aantal_woningen_vlaanderen` / `aantal_woningen_brussel` (vaste percentages) tot echte stockdata beschikbaar zijn.

Per stock: `aantal_<stock>_vlaanderen_{jaar}`, `_brussel_{jaar}`, `_totaal_{jaar}`. De simulatie leest enkel `*_totaal_*`.


In [34]:
# Voorbeeldverdeling vastgoed per regio — te vervangen door echte data
PCT_GEÏSOLEERD = 0.20
PCT_NIET_GEÏSOLEERD = 0.80
PCT_ONBEBOUWDE_BEBOUWBAAR = 0.20

for df in (df_lden, df_lnight):
    won_v = df["aantal_woningen_vlaanderen"]
    won_b = df["aantal_woningen_brussel"]

    maak_regio_kolommen(
        df,
        "aantal_bewoonde_geïsoleerde_huizen",
        BEGINJAAR,
        won_v * PCT_GEÏSOLEERD,
        won_b * PCT_GEÏSOLEERD,
    )
    maak_regio_kolommen(
        df,
        "aantal_bewoonde_niet_geïsoleerde_huizen",
        BEGINJAAR,
        won_v * PCT_NIET_GEÏSOLEERD,
        won_b * PCT_NIET_GEÏSOLEERD,
    )
    maak_regio_kolommen(
        df,
        "aantal_onbebouwde_bebouwbare_percelen",
        BEGINJAAR,
        won_v * PCT_ONBEBOUWDE_BEBOUWBAAR,
        won_b * PCT_ONBEBOUWDE_BEBOUWBAAR,
    )
    maak_regio_kolommen(
        df,
        "aantal_onbebouwde_onbebouwbare_percelen",
        BEGINJAAR,
        df["aantal_niet_bebouwbare_percelen_woongebied_schrapping"],
        0.0,
    )
    maak_regio_kolommen(
        df,
        "aantal_perceel_eigendom_overheid",
        BEGINJAAR,
        0.0,
        0.0,
    )
    maak_regio_kolommen(
        df,
        "aantal_woning_eigendom_overheid",
        BEGINJAAR,
        0.0,
        0.0,
    )

## Grafieken maken

Grafieken per geluidscontour voor **Lden** en **Lnight**, met het Idea Consult Altair-thema (`idea_consult_altair_theme.py`). Alles als staafdiagram.

Per indicator, drie grafieken:
1. **Totaal** — gegroepeerde staafgrafiek (Inwoners | Ernstig gehinderden)
2. **Inwoners** — gestapelde staafgrafiek (Vlaanderen + Brussel)
3. **Ernstig gehinderden** — gestapelde staafgrafiek (Vlaanderen + Brussel)

Voer eerst de cellen over samenvoegen, dosis–effect en `aantal_ernstig_gehinderden_*_2026` uit.


In [35]:
import altair as alt
import pandas as pd

import idea_consult_altair_theme  # noqa: F401 — registreert en activeert het Idea Consult-thema

from idea_consult_altair_theme import CATEGORY_COLORS

# --- As en hulp (x-as per indicator: geluidscontour-banden) ---
grafiek_id_kolommen = [
    "geluidscontour",
    "db_ondergrens",
    "db_bovengrens",
    "db_midden",
]


def contour_x(df: pd.DataFrame) -> alt.X:
    """X-as: 1 dB-contourbanden gesorteerd op geluidsniveau."""
    return alt.X(
        "geluidscontour:N",
        title="Geluidscontour (dB)",
        sort=alt.EncodingSortField(field="db_midden", order="ascending"),
        axis=alt.Axis(labelAngle=-45),
    )


regio_kleuren = alt.Scale(
    domain=["Vlaanderen", "Brussel"],
    range=CATEGORY_COLORS[:2],
)

regio_tooltips = [
    alt.Tooltip("geluidscontour:N", title="Contour"),
    alt.Tooltip("db_ondergrens:Q", title="dB ondergrens", format="d"),
    alt.Tooltip("db_bovengrens:Q", title="dB bovengrens", format="d"),
    alt.Tooltip("db_midden:Q", title="dB midden", format=".1f"),
    alt.Tooltip("regio:N", title="Regio"),
    alt.Tooltip("aantal:Q", title="Aantal", format=",.0f"),
]


def gestapelde_regio_staafgrafiek(
    df: pd.DataFrame,
    kolommen: dict[str, str],
    titel: str,
    y_titel: str = "Aantal personen",
) -> alt.Chart:
    """Gestapelde staafgrafiek: Vlaanderen + Brussel per contourband."""
    data = (
        df[grafiek_id_kolommen + list(kolommen.keys())]
        .sort_values("db_midden")
        .melt(
            id_vars=grafiek_id_kolommen,
            value_vars=list(kolommen.keys()),
            var_name="regio_kolom",
            value_name="aantal",
        )
    )
    data["regio"] = data["regio_kolom"].map(kolommen)

    return (
        alt.Chart(data)
        .mark_bar()
        .encode(
            x=contour_x(df),
            y=alt.Y("aantal:Q", stack="zero", title=y_titel, axis=alt.Axis(format=",.0f")),
            color=alt.Color(
                "regio:N",
                title="Regio",
                scale=regio_kleuren,
                sort=["Vlaanderen", "Brussel"],
            ),
            order=alt.Order("regio:N", sort="ascending"),
            tooltip=regio_tooltips,
        )
        .properties(title=titel, height=420)
    )


totaal_tooltips = [
    alt.Tooltip("geluidscontour:N", title="Contour"),
    alt.Tooltip("db_ondergrens:Q", title="dB ondergrens", format="d"),
    alt.Tooltip("db_bovengrens:Q", title="dB bovengrens", format="d"),
    alt.Tooltip("db_midden:Q", title="dB midden", format=".1f"),
    alt.Tooltip("reeks:N", title="Reeks"),
    alt.Tooltip("aantal:Q", title="Aantal", format=",.0f"),
]


def maak_grafiek_set(df: pd.DataFrame, indicator: str) -> dict[str, alt.Chart]:
    """Bouw de drie contourgrafieken voor Lden of Lnight."""
    grafiek_data = (
        df[grafiek_id_kolommen + ["inwoners", "aantal_ernstig_gehinderden_totaal_2026"]]
        .sort_values("db_midden")
        .melt(
            id_vars=grafiek_id_kolommen,
            value_vars=["inwoners", "aantal_ernstig_gehinderden_totaal_2026"],
            var_name="reeks",
            value_name="aantal",
        )
    )
    grafiek_data["reeks"] = grafiek_data["reeks"].map(
        {
            "inwoners": "Inwoners",
            "aantal_ernstig_gehinderden_totaal_2026": "Ernstig gehinderden",
        }
    )

    totaal = (
        alt.Chart(grafiek_data)
        .mark_bar()
        .encode(
            x=contour_x(df),
            xOffset=alt.XOffset(
                "reeks:N",
                sort=["Inwoners", "Ernstig gehinderden"],
            ),
            y=alt.Y("aantal:Q", title="Aantal personen", axis=alt.Axis(format=",.0f")),
            color=alt.Color(
                "reeks:N",
                title=None,
                scale=alt.Scale(
                    domain=["Inwoners", "Ernstig gehinderden"],
                    range=CATEGORY_COLORS[:2],
                ),
            ),
            tooltip=totaal_tooltips,
        )
        .properties(
            title=f"Inwoners en ernstig gehinderden per geluidscontour ({indicator})",
            height=420,
        )
    )

    inwoners_gestapeld = gestapelde_regio_staafgrafiek(
        df,
        {"inwoners_vlaanderen": "Vlaanderen", "inwoners_brussel": "Brussel"},
        titel=f"Inwoners per geluidscontour ({indicator}) — Vlaanderen en Brussel",
    )

    gehinderden_gestapeld = gestapelde_regio_staafgrafiek(
        df,
        {
            "aantal_ernstig_gehinderden_vlaanderen_2026": "Vlaanderen",
            "aantal_ernstig_gehinderden_brussel_2026": "Brussel",
        },
        titel=f"Ernstig gehinderden per geluidscontour ({indicator}) — Vlaanderen en Brussel",
    )

    return {
        "totaal": totaal,
        "inwoners_gestapeld": inwoners_gestapeld,
        "gehinderden_gestapeld": gehinderden_gestapeld,
    }



grafieken_lden = maak_grafiek_set(df_lden, "Lden")
grafieken_lnight = maak_grafiek_set(df_lnight, "Lnight")

# Voorbeeld in notebook: Lden-totaal (x-as 45–75); Lnight heeft eigen bereik (bijv. 40–70)

# Preview Lden-grafieken (staafdiagrammen)
grafieken_lden["totaal"]
grafieken_lden["inwoners_gestapeld"]
grafieken_lden["gehinderden_gestapeld"]


alt.Chart(...)

## Grafieken exporteren naar PNG

Zes losse PNG’s in `output/grafieken/` (`lden_*.png` en `lnight_*.png`, elk drie grafieken). Geen gecombineerde PNG’s. Lden en Lnight hebben elk een eigen x-asbereik (eigen onder-/bovengrens). Vereiste: `pip install vl-convert-python` (staat in `requirements.txt`).

In [36]:
from pathlib import Path

GRAFIEKEN_MAP = Path("output/grafieken")
GRAFIEKEN_MAP.mkdir(parents=True, exist_ok=True)


def exporteer_grafiek_naar_png(
    chart: alt.Chart,
    bestandsnaam: str,
    scale_factor: float = 2,
) -> Path:
    """Sla een Altair-grafiek op als PNG (via vl-convert)."""
    pad = GRAFIEKEN_MAP / bestandsnaam
    chart.save(str(pad), scale_factor=scale_factor)
    print(f"Opgeslagen: {pad.resolve()}")
    return pad


grafieken_te_exporteren: dict[str, alt.Chart] = {}
for prefix, grafiek_set in [
    ("lden", grafieken_lden),
    ("lnight", grafieken_lnight),
]:
    for sleutel, grafiek in grafiek_set.items():
        grafieken_te_exporteren[f"{prefix}_{sleutel}.png"] = grafiek

for naam, grafiek in grafieken_te_exporteren.items():
    exporteer_grafiek_naar_png(grafiek, naam)

Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_totaal.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_inwoners_gestapeld.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lden_gehinderden_gestapeld.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Financiering van de maatregelen ‘RO en beheer’ als 2e pijler van de Ba)\dashboard-balanced-approach\output\grafieken\lnight_totaal.png
Opgeslagen: C:\Users\AlexanderL\OneDrive - Idea Consult nv\RTD\2. Projects\25 RTD 72  (Fina

## 7. Exporteren naar het dashboard

De dataframes worden weggeschreven naar `input/`. Het dashboard koppelt zones via `db_midden` en leest stockkolommen `aantal_<stock>_totaal_{BEGINJAAR}` (regionale opsplitsing blijft op het contour).


db_contour,lower,upper,inwoners,huizen,aantal bebouwbare percelen die werden gecreëerd door woongebieden aan te duiden,aantal niet-bebouwbarepercelen die worden gecreëerd door woongebied te schrappen,prijs_onbebouwde_bebouwbare_percelen,prijs_onbebouwde_onbebouwbare_percelen,prijs_bewoonde_niet_geïsoleerde_woning,prijs_bewoonde_geïsoleerde_woning,midden,dosis_effect_relatie,gemiddeld_aantal_inwoners_per_huis,aantal_ernstig_gehinderden_2026,aantal_bewoonde_geïsoleerde_huizen_2026,aantal_bewoonde_niet_geïsoleerde_huizen_2026,aantal_onbebouwde_bebouwbare_percelen_2026,aantal_perceel_eigendom_overheid_2026,aantal_woning_eigendom_overheid

In [37]:
df_lden.columns

Index(['geluidscontour', 'db_ondergrens', 'db_bovengrens', 'inwoners',
       'aantal_bebouwbare_percelen_woongebied',
       'aantal_niet_bebouwbare_percelen_woongebied_schrapping',
       'prijs_onbebouwde_bebouwbare_percelen',
       'prijs_onbebouwde_onbebouwbare_percelen',
       'prijs_bewoonde_niet_geïsoleerde_woning',
       'prijs_bewoonde_geïsoleerde_woning',
       'gemiddeld_aantal_inwoners_per_huis', 'inwoners_vlaanderen',
       'inwoners_brussel', 'aantal_woningen_vlaanderen',
       'aantal_woningen_brussel', 'aantal_woningen_totaal', 'db_midden',
       'dosis_effect_relatie', 'aantal_ernstig_gehinderden_vlaanderen_2026',
       'aantal_ernstig_gehinderden_brussel_2026',
       'aantal_ernstig_gehinderden_totaal_2026',
       'aantal_bewoonde_geïsoleerde_huizen_vlaanderen_2026',
       'aantal_bewoonde_geïsoleerde_huizen_brussel_2026',
       'aantal_bewoonde_geïsoleerde_huizen_totaal_2026',
       'aantal_bewoonde_niet_geïsoleerde_huizen_vlaanderen_2026',
       'aant

In [38]:
df_lden.to_csv("input/lden_contour.csv")
df_lnight.to_csv("input/lnight_contour.csv")

## 8. Intensiteit maatregelstromen (flows)

Per maatregel berekenen we in- en uitstroomintensiteit per contour. Onderstaande formules zijn een **eerste schets** (woongebiedverbod / aankoopbeleid); de noemer gebruikt voorlopig de indicator voor niet-bebouwbare percelen bij woongebiedschrapping.


In [39]:
# Woongebiedverbod — intensiteit = aandeel bebouwbare percelen / referentiestock
""" _bebouwbaar = df_lden["aantal_bebouwbare_percelen_woongebied"]
_referentie = df_lden["aantal_onbebouwde_onbebouwbare_percelen_totaal_2026"]

df_lden["woongebied_verbod_inflow_zonder_maatregel"] = _bebouwbaar / _referentie
df_lden["woongebied_verbod_outflow_zonder_maatregel"] = _bebouwbaar / _referentie
df_lden["woongebied_verbod_inflow_met_maatregel"] = 0
df_lden["woongebied_verbod_outflow_met_maatregel"] = 0

# Aankoopbeleid (voorbeeld — zelfde structuur, andere maatregel)
df_lden["aankoopbeleid_inflow_zonder_maatregel"] = 0
df_lden["aankoopbeleid_outflow_zonder_maatregel"] = 0
df_lden["aankoopbeleid_inflow_met_maatregel"] = _bebouwbaar / _referentie
df_lden["aankoopbeleid_outflow_met_maatregel"] = _bebouwbaar / _referentie """

' _bebouwbaar = df_lden["aantal_bebouwbare_percelen_woongebied"]\n_referentie = df_lden["aantal_onbebouwde_onbebouwbare_percelen_totaal_2026"]\n\ndf_lden["woongebied_verbod_inflow_zonder_maatregel"] = _bebouwbaar / _referentie\ndf_lden["woongebied_verbod_outflow_zonder_maatregel"] = _bebouwbaar / _referentie\ndf_lden["woongebied_verbod_inflow_met_maatregel"] = 0\ndf_lden["woongebied_verbod_outflow_met_maatregel"] = 0\n\n# Aankoopbeleid (voorbeeld — zelfde structuur, andere maatregel)\ndf_lden["aankoopbeleid_inflow_zonder_maatregel"] = 0\ndf_lden["aankoopbeleid_outflow_zonder_maatregel"] = 0\ndf_lden["aankoopbeleid_inflow_met_maatregel"] = _bebouwbaar / _referentie\ndf_lden["aankoopbeleid_outflow_met_maatregel"] = _bebouwbaar / _referentie '

## 9. Exploratie omgevingsloket-data (juni 2026)

Drie nieuwe bronbestanden uit `data/` met vergunningen/projecten per gemeente en jaar. Ze hebben een **pivot-achtige Excel-structuur** (geen platte tabel): rij 0–2 zijn koppen, data vanaf rij 3.

| Bestand | Rijen | Kolommen | Handelingen (kolomgroepen) |
|---------|------:|---------:|--------------------------|
| `vergunningen_omgevingsloket_2026.xlsx` | 219 | 116 | Totalen, -, Nieuwbouw, Sloop, Verbouwen of hergebruik |
| `vergunningen_kwetsbare_functies_2026.xlsx` | 75 | 20 | Totalen, - |
| `vergunningen_verkaveling_2026.xlsx` | 152 | 32 | Totalen, -, Sloop |

**Gemeenschappelijke rij-dimensies**
- Kolom 0: `Jaar indiening` (2019–2026 + totaal)
- Kolom 1: `Project Gemeente` (gemeentenaam of `Totalen`)

**Gemeenschappelijke metrieken** (per handeling × gebouwfuctie, elk 6 kolommen):
1. Aantal projecten
2. Aantal gebouwen
3. Aantal wooneenheden
4. Nuttige woonoppervlakte
5. Bovengronds nuttige oppervlakte
6. Bovengronds grond oppervlakte

**Extra bij omgevingsloket** — gebouwfucties onder Nieuwbouw / Verbouwen: `andere`, `eengezinswoning`, `kamerwoning`, `eengezins- en kamerwoning`, `meergezinswoning`, `wonen`.

In [40]:
import re
from pathlib import Path

OMGEVINGSLOKET_BESTANDEN = {
    "vergunningen_omgevingsloket_2026": Path("data/vergunningen_omgevingsloket_2026.xlsx"),
    "vergunningen_kwetsbare_functies_2026": Path("data/vergunningen_kwetsbare_functies_2026.xlsx"),
    "vergunningen_verkaveling_2026": Path("data/vergunningen_verkaveling_2026.xlsx"),
}

METRIEKEN = [
    "Aantal projecten",
    "Aantal gebouwen",
    "Aantal wooneenheden",
    "Nuttige woonoppervlakte",
    "Bovengronds nuttige oppervlakte",
    "Bovengronds grond oppervlakte",
]


def lees_omgevingsloket_raw(pad: Path) -> pd.DataFrame:
    """Lees Excel zonder header; eerste 3 rijen zijn pivot-koppen."""
    return pd.read_excel(pad, sheet_name=0, header=None)


def kolomstructuur(raw: pd.DataFrame) -> pd.DataFrame:
    """Zet hiërarchische kolomkoppen om naar één overzichtstabel."""
    rij_handeling = raw.iloc[0].fillna("")
    rij_functie = raw.iloc[1].fillna("")
    rij_metriek = raw.iloc[2].fillna("")

    records = []
    for col in range(2, raw.shape[1]):
        records.append(
            {
                "kolom_index": col,
                "handeling": str(rij_handeling[col]),
                "gebouw_functie": str(rij_functie[col]),
                "metriek": str(rij_metriek[col]),
            }
        )
    return pd.DataFrame(records)


def parse_omgevingsloket_waarde(waarde):
    """Zet Belgische getallen en oppervlaktes (bv. '1.234,56 m²') om naar float."""
    if pd.isna(waarde):
        return pd.NA
    if isinstance(waarde, (int, float)) and not isinstance(waarde, bool):
        return float(waarde)

    tekst = str(waarde).strip()
    if not tekst or tekst.lower() == "nan":
        return pd.NA

    tekst = re.sub(r"\s*m.?$", "", tekst, flags=re.IGNORECASE).strip()
    if "," in tekst:
        tekst = tekst.replace(".", "").replace(",", ".")
    try:
        return float(tekst)
    except ValueError:
        return pd.NA


def normaliseer_jaar_indiening(waarde) -> int | str:
    tekst = str(waarde).strip()
    if tekst == "Totalen":
        return "Totalen"
    try:
        return int(tekst)
    except ValueError:
        return tekst


def pivot_naar_lang(raw: pd.DataFrame, bron: str) -> pd.DataFrame:
    """Zet pivot-Excel om naar lange tabel: jaar × gemeente × handeling × functie × metriek."""
    rij_handeling = raw.iloc[0].fillna("")
    rij_functie = raw.iloc[1].fillna("")
    rij_metriek = raw.iloc[2].fillna("")

    records: list[dict] = []
    for row_idx in range(3, len(raw)):
        jaar_raw = str(raw.iloc[row_idx, 0]).strip()
        if jaar_raw in ("nan", "Jaar indiening"):
            continue

        gemeente_raw = raw.iloc[row_idx, 1]
        gemeente = pd.NA if pd.isna(gemeente_raw) else str(gemeente_raw).strip()

        for col in range(2, raw.shape[1]):
            metriek = str(rij_metriek[col]).strip()
            if not metriek or metriek == "nan":
                continue

            handeling = str(rij_handeling[col]).strip() or pd.NA
            gebouw_functie = str(rij_functie[col]).strip() or pd.NA
            waarde = parse_omgevingsloket_waarde(raw.iloc[row_idx, col])

            records.append(
                {
                    "bron": bron,
                    "jaar_indiening": normaliseer_jaar_indiening(jaar_raw),
                    "gemeente": gemeente,
                    "handeling": handeling,
                    "gebouw_functie": gebouw_functie,
                    "metriek": metriek,
                    "waarde": waarde,
                }
            )

    return pd.DataFrame(records)


def rijstructuur(raw: pd.DataFrame) -> pd.DataFrame:
    """Overzicht rij-dimensies (jaar × gemeente)."""
    data = raw.iloc[3:].copy()
    data.columns = ["jaar_indiening", "gemeente", *range(2, raw.shape[1])]
    return (
        data[["jaar_indiening", "gemeente"]]
        .dropna(how="all")
        .assign(
            jaar_indiening=lambda d: d["jaar_indiening"].astype(str).str.strip(),
            gemeente=lambda d: d["gemeente"].astype(str).str.strip(),
        )
    )


def verken_omgevingsloket_bestand(naam: str, pad: Path) -> None:
    print("=" * 80)
    print(naam)
    print(pad)
    print("=" * 80)

    raw = lees_omgevingsloket_raw(pad)
    print(f"Vorm: {raw.shape[0]} rijen × {raw.shape[1]} kolommen (incl. koppen)")

    kolommen = kolomstructuur(raw)
    print("\nKolomgroepen (handeling × gebouwfuctie × metriek):")
    display(
        kolommen.groupby(["handeling", "gebouw_functie"], dropna=False)
        .size()
        .reset_index(name="aantal_metrieken")
        .sort_values(["handeling", "gebouw_functie"])
    )

    print("\nUnieke metrieken:")
    print(kolommen["metriek"].unique().tolist())

    rijen = rijstructuur(raw)
    jaren = sorted(x for x in rijen["jaar_indiening"].unique() if x not in ("nan", "Jaar indiening"))
    gemeenten = sorted(
        g for g in rijen["gemeente"].dropna().unique() if g not in ("nan", "Project Gemeente", "Totalen")
    )
    print(f"\nJaren: {jaren}")
    print(f"Aantal gemeenten: {len(gemeenten)}")
    print("Gemeenten:", ", ".join(gemeenten[:12]), ("..." if len(gemeenten) > 12 else ""))

    print("\nVoorbeeld data (eerste gemeente, eerste 6 metrieken):")
    voorbeeld = raw.iloc[4, :8]
    print(voorbeeld.to_string())


for naam, pad in OMGEVINGSLOKET_BESTANDEN.items():
    verken_omgevingsloket_bestand(naam, pad)

vergunningen_omgevingsloket_2026
data\vergunningen_omgevingsloket_2026.xlsx
Vorm: 221 rijen × 116 kolommen (incl. koppen)

Kolomgroepen (handeling × gebouwfuctie × metriek):


,handeling,gebouw_functie,aantal_metrieken
0,-,-,6
1,-,Totalen,6
2,Nieuwbouw,-,6
3,Nieuwbouw,Totalen,6
4,Nieuwbouw,andere,6
5,Nieuwbouw,eengezinswoning,6
6,Nieuwbouw,kamerwoning,6
7,Nieuwbouw,meergezinswoning,6
8,Sloop,-,6
9,Sloop,Totalen,6



Unieke metrieken:
['Aantal projecten', 'Aantal gebouwen', 'Aantal wooneenheden', 'Nuttige woonoppervlakte', 'Bovengronds nuttige oppervlakte', 'Bovengronds grond oppervlakte']

Jaren: ['2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', 'Totalen']
Aantal gemeenten: 47
Gemeenten: (deels) Niet in Vlaanderen, Aarschot, Asse, Begijnendijk, Bertem, Beveren-Kruibeke-Zwijndrecht, Bonheiden, Boortmeerbeek, Bornem, Buggenhout, Diest, Dilbeek ...

Voorbeeld data (eerste gemeente, eerste 6 metrieken):
0        2019
1         NaN
2           3
3           1
4           0
5       0,00 
6    36,00 m²
7    36,00 m²
vergunningen_kwetsbare_functies_2026
data\vergunningen_kwetsbare_functies_2026.xlsx
Vorm: 77 rijen × 20 kolommen (incl. koppen)

Kolomgroepen (handeling × gebouwfuctie × metriek):


,handeling,gebouw_functie,aantal_metrieken
0,-,-,6
1,-,Totalen,6
2,Totalen,,6



Unieke metrieken:
['Aantal projecten', 'Aantal gebouwen', 'Aantal wooneenheden', 'Nuttige woonoppervlakte', 'Bovengronds nuttige oppervlakte', 'Bovengronds grond oppervlakte']

Jaren: ['2020', '2021', '2022', '2023', '2024', '2025', '2026', 'Totalen']
Aantal gemeenten: 26
Gemeenten: Aarschot, Asse, Begijnendijk, Boortmeerbeek, Dilbeek, Grimbergen, Haacht, Herent, Holsbeek, Huldenberg, Kampenhout, Kortenberg ...

Voorbeeld data (eerste gemeente, eerste 6 metrieken):
0     2020
1      NaN
2        9
3        0
4        0
5    0,00 
6    0,00 
7    0,00 
vergunningen_verkaveling_2026
data\vergunningen_verkaveling_2026.xlsx
Vorm: 154 rijen × 32 kolommen (incl. koppen)

Kolomgroepen (handeling × gebouwfuctie × metriek):


,handeling,gebouw_functie,aantal_metrieken
0,-,-,6
1,-,Totalen,6
2,Sloop,-,6
3,Sloop,Totalen,6
4,Totalen,,6



Unieke metrieken:
['Aantal projecten', 'Aantal gebouwen', 'Aantal wooneenheden', 'Nuttige woonoppervlakte', 'Bovengronds nuttige oppervlakte', 'Bovengronds grond oppervlakte']

Jaren: ['2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', 'Totalen']
Aantal gemeenten: 33
Gemeenten: Aarschot, Asse, Begijnendijk, Bertem, Bonheiden, Boortmeerbeek, Dilbeek, Grimbergen, Haacht, Herent, Hoeilaart, Holsbeek ...

Voorbeeld data (eerste gemeente, eerste 6 metrieken):
0     2019
1      NaN
2        2
3        0
4        0
5    0,00 
6    0,00 
7    0,00 


### 9.1 Pivot naar lange tabel

Kolommen in de lange tabel: `bron`, `jaar_indiening`, `gemeente`, `handeling`, `gebouw_functie`, `metriek`, `waarde`.

Export naar `data/*_lang.csv` (puntkomma-gescheiden).

In [41]:
omg_lang: dict[str, pd.DataFrame] = {}

for naam, pad in OMGEVINGSLOKET_BESTANDEN.items():
    raw = lees_omgevingsloket_raw(pad)
    df_lang = pivot_naar_lang(raw, bron=naam)
    omg_lang[naam] = df_lang

    export_pad = pad.with_name(f"{pad.stem}_lang.csv")
    df_lang.to_csv(export_pad, sep=";", index=False)

    print(f"{naam}: {len(df_lang):,} rijen → {export_pad}")
    display(df_lang.head(8))

# Gecombineerd overzicht
df_omgevingsloket_lang = pd.concat(omg_lang.values(), ignore_index=True)
print(f"\nTotaal gecombineerd: {len(df_omgevingsloket_lang):,} rijen")
display(
    df_omgevingsloket_lang.groupby(["bron", "metriek"], dropna=False)["waarde"]
    .apply(lambda s: s.notna().sum())
    .reset_index(name="aantal_ingevuld")
)

vergunningen_omgevingsloket_2026: 24,852 rijen → data\vergunningen_omgevingsloket_2026_lang.csv


,bron,jaar_indiening,gemeente,handeling,gebouw_functie,metriek,waarde
0,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Aantal projecten,14597.0
1,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Aantal gebouwen,12689.0
2,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Aantal wooneenheden,14651.0
3,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Nuttige woonoppervlakte,2970567.27
4,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Bovengronds nuttige oppervlakte,5801978.89
5,vergunningen_omgevingsloket_2026,Totalen,NaN,Totalen,NaN,Bovengronds grond oppervlakte,3494779.71
6,vergunningen_omgevingsloket_2026,Totalen,NaN,-,Totalen,Aantal projecten,5663.0
7,vergunningen_omgevingsloket_2026,Totalen,NaN,-,Totalen,Aantal gebouwen,0.0


vergunningen_kwetsbare_functies_2026: 1,332 rijen → data\vergunningen_kwetsbare_functies_2026_lang.csv


,bron,jaar_indiening,gemeente,handeling,gebouw_functie,metriek,waarde
0,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Aantal projecten,139.0
1,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Aantal gebouwen,0.0
2,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Aantal wooneenheden,0.0
3,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Nuttige woonoppervlakte,0.0
4,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Bovengronds nuttige oppervlakte,0.0
5,vergunningen_kwetsbare_functies_2026,Totalen,NaN,Totalen,NaN,Bovengronds grond oppervlakte,0.0
6,vergunningen_kwetsbare_functies_2026,Totalen,NaN,-,Totalen,Aantal projecten,139.0
7,vergunningen_kwetsbare_functies_2026,Totalen,NaN,-,Totalen,Aantal gebouwen,0.0


vergunningen_verkaveling_2026: 4,530 rijen → data\vergunningen_verkaveling_2026_lang.csv


,bron,jaar_indiening,gemeente,handeling,gebouw_functie,metriek,waarde
0,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Aantal projecten,624.0
1,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Aantal gebouwen,238.0
2,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Aantal wooneenheden,0.0
3,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Nuttige woonoppervlakte,0.0
4,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Bovengronds nuttige oppervlakte,0.0
5,vergunningen_verkaveling_2026,Totalen,NaN,Totalen,NaN,Bovengronds grond oppervlakte,0.0
6,vergunningen_verkaveling_2026,Totalen,NaN,-,Totalen,Aantal projecten,497.0
7,vergunningen_verkaveling_2026,Totalen,NaN,-,Totalen,Aantal gebouwen,0.0



Totaal gecombineerd: 30,714 rijen


,bron,metriek,aantal_ingevuld
0,vergunningen_kwetsbare_functies_2026,Aantal gebouwen,222
1,vergunningen_kwetsbare_functies_2026,Aantal projecten,222
2,vergunningen_kwetsbare_functies_2026,Aantal wooneenheden,222
3,vergunningen_kwetsbare_functies_2026,Bovengronds grond oppervlakte,222
4,vergunningen_kwetsbare_functies_2026,Bovengronds nuttige oppervlakte,222
5,vergunningen_kwetsbare_functies_2026,Nuttige woonoppervlakte,222
6,vergunningen_omgevingsloket_2026,Aantal gebouwen,2216
7,vergunningen_omgevingsloket_2026,Aantal projecten,2216
8,vergunningen_omgevingsloket_2026,Aantal wooneenheden,2216
9,vergunningen_omgevingsloket_2026,Bovengronds grond oppervlakte,2216


## 10. Conversietabel gemeente → geluidscontour (dB)

Vergunningsdata zijn geleverd per **gemeente**. Voor Brussel kunnen we omzetten naar 1 dB-contouren via `inwoners_brussel_sector_contour_2024.xlsx`.

**Methode:** kolom `Part de la surface du qs dans le noise contour` (= aandeel van de **statistische sector** in die dB-contour; *qs* is typo voor *ss*). Per gemeente sommeren we deze aandelen over alle sectoren:

$$\text{aandeel}_{g,d} = \frac{\sum_{s \in g} \text{opp-aandeel}_{s,d}}{\sum_{d'} \sum_{s \in g} \text{opp-aandeel}_{s,d'}}$$

De noemer normaliseert tot 1 per gemeente. Enkelvoudige `dB` uit Brussel koppelen we aan de landelijke band waar `db_ondergrens == dB`.

**Beperking:** deze tabel geldt voor **Brusselse gemeenten** (19 gemeenten in de sectorfile). Vlaamse ringgemeenten in de vergunningsdata hebben (nog) geen sector-contour-gewichten — zie §11.

In [42]:
BRUSSEL_PAD = Path("data/inwoners_brussel_sector_contour_2024.xlsx")
OPP_AANDEEL_KOLOM = "Part de la surface du qs dans le noise contour"
GEMEENTE_KOLOM = "T_MUN_NL"


def conversietabel_gemeente_naar_db(
    brussel_sector: pd.DataFrame,
    gemeente_kolom: str = GEMEENTE_KOLOM,
) -> pd.DataFrame:
    """Gemeente × dB met genormaliseerd ruimtelijk aandeel (som sector-opp-aandelen)."""
    gewicht = (
        brussel_sector.groupby([gemeente_kolom, "dB"], as_index=False)[OPP_AANDEEL_KOLOM]
        .sum()
        .rename(columns={gemeente_kolom: "gemeente", OPP_AANDEEL_KOLOM: "gewicht_ruimtelijk"})
    )
    gewicht["aandeel"] = gewicht["gewicht_ruimtelijk"] / gewicht.groupby("gemeente")["gewicht_ruimtelijk"].transform("sum")
    return gewicht.rename(columns={"dB": "db"})


def koppel_conversie_aan_contourband(
    conversie: pd.DataFrame,
    df_contour: pd.DataFrame,
) -> pd.DataFrame:
    """Voeg geluidscontour-label toe via db_ondergrens."""
    banden = df_contour[["geluidscontour", "db_ondergrens", "db_bovengrens"]].drop_duplicates()
    return conversie.merge(banden, left_on="db", right_on="db_ondergrens", how="left")


brussel_lden_ruw = pd.read_excel(BRUSSEL_PAD, sheet_name="lden")
brussel_lnight_ruw = pd.read_excel(BRUSSEL_PAD, sheet_name="lnight")

conversie_lden = conversietabel_gemeente_naar_db(brussel_lden_ruw)
conversie_lnight = conversietabel_gemeente_naar_db(brussel_lnight_ruw)

conversie_lden_band = koppel_conversie_aan_contourband(conversie_lden, df_lden)
conversie_lnight_band = koppel_conversie_aan_contourband(conversie_lnight, df_lnight)

print(f"Lden: {conversie_lden['gemeente'].nunique()} gemeenten, dB {conversie_lden['db'].min()}–{conversie_lden['db'].max()}")
print(f"Lnight: {conversie_lnight['gemeente'].nunique()} gemeenten, dB {conversie_lnight['db'].min()}–{conversie_lnight['db'].max()}")
display(conversie_lden_band.sort_values(["gemeente", "db"]).head(12))

Lden: 17 gemeenten, dB 45–64
Lnight: 13 gemeenten, dB 40–54


,gemeente,db,gewicht_ruimtelijk,aandeel,geluidscontour,db_ondergrens,db_bovengrens
0,Anderlecht,45,17.522141,0.399496,45-46,45,46
1,Anderlecht,46,20.698474,0.471915,46-47,46,47
2,Anderlecht,47,5.640015,0.128589,47-48,47,48
3,Brussel,45,11.156192,0.140722,45-46,45,46
4,Brussel,46,9.028959,0.113890,46-47,46,47
5,Brussel,47,4.640330,0.058532,47-48,47,48
6,Brussel,48,5.171395,0.065231,48-49,48,49
7,Brussel,49,8.447698,0.106558,49-50,49,50
8,Brussel,50,10.786538,0.136059,50-51,50,51
9,Brussel,51,7.707997,0.097227,51-52,51,52


## 11. Vergunningen van gemeente naar contour

Gemeentelijke tellers uit de lange vergunningstabellen verdelen we over contourbanden met de conversietabel uit §10:

`waarde_contour = waarde_gemeente × aandeel_{gemeente, dB}`

Daarna aggregeren we per `geluidscontour` (en behouden dimensies: jaar, handeling, gebouwfuctie, metriek).

**Let op:** de huidige vergunningsdata bevatten vooral **Vlaamse ringgemeenten** rond de luchthaven. Die staan niet in de Brusselse sectorfile → ze krijgen geen gewicht in deze conversie. Voor volledige flow-berekening is een analoge sector-contour-tabel voor Vlaanderen nodig, of een ruimtelijke koppeling gemeente↔contour.

In [43]:
def verdeel_gemeente_naar_contour(
    df_lang: pd.DataFrame,
    conversie_band: pd.DataFrame,
    jaartal: str | int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Verdeel gemeente-waarden over contourbanden; retourneer (contour, niet_toewijsbaar)."""
    df = df_lang.copy()
    df = df[df["gemeente"].notna() & ~df["gemeente"].isin(["-", "Totalen", ""])]
    if jaartal is not None:
        df = df[df["jaar_indiening"] == str(jaartal)]

    gekoppeld = df.merge(
        conversie_band[["gemeente", "db", "aandeel", "geluidscontour", "db_ondergrens"]],
        on="gemeente",
        how="left",
    )
    gekoppeld["waarde_contour"] = gekoppeld["waarde"] * gekoppeld["aandeel"]

    niet_toewijsbaar = (
        gekoppeld[gekoppeld["geluidscontour"].isna()]
        .groupby(["bron", "gemeente", "jaar_indiening", "handeling", "gebouw_functie", "metriek"], dropna=False)["waarde"]
        .sum()
        .reset_index()
    )

    dim_cols = ["bron", "geluidscontour", "db_ondergrens", "jaar_indiening", "handeling", "gebouw_functie", "metriek"]
    contour = (
        gekoppeld.dropna(subset=["geluidscontour"])
        .groupby(dim_cols, dropna=False)["waarde_contour"]
        .sum()
        .reset_index()
        .rename(columns={"waarde_contour": "waarde"})
    )
    return contour, niet_toewijsbaar


df_verg_lang = pd.read_csv("data/vergunningen_omgevingsloket_2026_lang.csv", sep=";")
verg_contour_lden, verg_niet_lden = verdeel_gemeente_naar_contour(
    df_verg_lang, conversie_lden_band, jaartal=2025
)

print("Voorbeeld: Nieuwbouw wooneenheden per contour (2025, Lden-gewichten Brussel)")
mask = (
    (verg_contour_lden["handeling"] == "Nieuwbouw")
    & (verg_contour_lden["metriek"] == "Aantal wooneenheden")
)
display(verg_contour_lden.loc[mask].sort_values("waarde", ascending=False).head(15))

print(f"\nNiet toewijsbaar (Vlaamse gemeenten): {verg_niet_lden['gemeente'].nunique()} gemeenten, "
      f"{verg_niet_lden['waarde'].sum():,.0f} eenheden totaal (2025)")
display(verg_niet_lden.groupby("gemeente")["waarde"].sum().sort_values(ascending=False).head(10))

Voorbeeld: Nieuwbouw wooneenheden per contour (2025, Lden-gewichten Brussel)


,bron,geluidscontour,db_ondergrens,jaar_indiening,handeling,gebouw_functie,metriek,waarde



Niet toewijsbaar (Vlaamse gemeenten): 32 gemeenten, 6,523,933 eenheden totaal (2025)


gemeente
Zaventem          1376267.04
Machelen           844001.15
Steenokkerzeel     623233.17
Kortenberg         523454.94
Vilvoorde          413707.63
Dilbeek            369876.33
Grimbergen         339494.44
Herent             242000.34
Tervuren           215895.53
Kraainem           197991.09
Name: waarde, dtype: float64